In [ ]:
import mo_gymnasium as mo_gym
import numpy as np
import envs
import csv
from src.modules.commun import Constant
from src.modules.Tools import Tools
from morl_baselines.multi_policy.envelope.envelope import Envelope

GAMMA = 1


# IMPORTANT____________________________________________________________________________________________
# Init Env by giving it the service querry to optimize + the folder where to find the PREPROCESSED data 
serviceIds = [0 , 1 , 2 ]
number_services = len(serviceIds)
MultiCloud_data_dir="./src/data/preprocessedData/NC_20_NS_50/NC_20_NS_50_01"
#______________________________________________________________________________________________________


GAMMA = 1
env = mo_gym.MORecordEpisodeStatistics(mo_gym.make("env/SelectService-pcn", preprocessed_data_dir=MultiCloud_data_dir,service_querry=serviceIds ), gamma=GAMMA)
eval_env = mo_gym.make("env/SelectService-pcn", preprocessed_data_dir=MultiCloud_data_dir,service_querry=serviceIds )


#### Test env with random actions just to make sure there is no bug in env and the MDP works fine

In [ ]:

nb_clouds  = env.multicloud.getNumberClouds()

obs, info = env.reset()  # Init the env
terminated = False
acc_rew = np.full(Constant.number_objectives, 0) # init acc reward
print("init obs: ",obs)
while (terminated == False):
    old_obs = obs
    action = env.action_space.sample()  # this is where you would insert your policy
    obs, reward, terminated, truncated, info = env.step(action)
    acc_rew = acc_rew + reward
    #print("-------------------------------")
    print("old obs:  ",old_obs,"\naction :", action,"\nnew obs: ",obs,"\nreward :", reward ,"\terminated :", terminated )
    print("_____________________________________________________")
print("acc_rew:",acc_rew)

#print("Selected services : ", env.composition.print_composition())

In [ ]:
pf = env.unwrapped.pareto_front()
print(pf)

In [ ]:
agent = Envelope(
        env,
        max_grad_norm=0.1,
        learning_rate=3e-3,
        gamma=GAMMA,
        batch_size=64,
        net_arch=[256, 256, 256, 256],
        buffer_size=int(2e6),
        initial_epsilon=1.0,
        final_epsilon=0.05,
        epsilon_decay_steps=50000,
        initial_homotopy_lambda=0.0,
        final_homotopy_lambda=1.0,
        homotopy_decay_steps=1000,
        learning_starts=100,
        envelope=True,
        gradient_updates=1,
        target_net_update_freq=1000,  # 1000,  # 500 reduce by gradient updates
        tau=1,
        log=True,
        project_name="Envelope_test",
        experiment_name="Envelope",
    )

In [ ]:
agent.train(
        total_timesteps=10000,
        total_episodes=None,
        weight=None,
        eval_env=eval_env,
        ref_point=np.array([-1 ,-1 ,-1 ,-1 ,-1 , -1]),
        known_pareto_front=pf,
        num_eval_weights_for_front=100,
        eval_freq=100,
        reset_num_timesteps=False,
        reset_learning_starts=False,
    )



# Use the trained agent with diffrent prefrences  :

In [ ]:

nb_clouds  = env.multicloud.getNumberClouds()

obs, info = env.reset()  # Init the env
terminated = False
acc_rew = np.full(Constant.number_objectives, 0) # init acc reward
print("init obs: ",obs)
while (terminated == False):
    old_obs = obs
    action = agent.eval( obs =obs, w = [1/6,1/6,1/6,1/6,1/6,1/6])
    obs, reward, terminated, truncated, info = env.step(action)
    acc_rew = acc_rew + reward
    #print("-------------------------------")
    print("old obs:  ",old_obs,"\naction :", action,"\nnew obs: ",obs,"\nreward :", reward ,"\terminated :", terminated )
    print("_____________________________________________________")
print("acc_rew:",acc_rew)
print("Selected services : ", env.composition.print_composition())


In [ ]:

# Second prefrences :
nb_clouds  = env.multicloud.getNumberClouds()

obs, info = env.reset()  # Init the env
terminated = False
acc_rew = np.full(Constant.number_objectives, 0) # init acc reward
print("init obs: service [",obs//nb_clouds ,"cloud ",obs%nb_clouds,"]")
while (terminated == False):
    old_obs = obs
    action = mp_moql.eval( obs = obs, w = [1/4,1/4,1/4,0,1/4,0])
    obs, reward, terminated, truncated, info = env.step(action)
    acc_rew = acc_rew + reward
    #print("-------------------------------")
    print("old obs: [ service",old_obs//nb_clouds,"cloud",old_obs%nb_clouds,"]\naction :", action,"\nnew obs: [ service",obs//nb_clouds,"cloud",obs%nb_clouds,"]\nreward :", reward ,"\terminated :", terminated )
    print("_____________________________________________________")
print("acc_rew:",acc_rew)

print("Selected services : ", env.composition.print_composition())

In [ ]:

# Second prefrences :
nb_clouds  = env.multicloud.getNumberClouds()

obs, info = env.reset()  # Init the env
terminated = False
acc_rew = np.full(Constant.number_objectives, 0) # init acc reward
print("init obs: service [",obs//nb_clouds ,"cloud ",obs%nb_clouds,"]")
while (terminated == False):
    old_obs = obs
    action = mp_moql.eval( obs = obs, w = [1/2,1/4,1/4,0,0,0])
    obs, reward, terminated, truncated, info = env.step(action)
    acc_rew = acc_rew + reward
    #print("-------------------------------")
    print("old obs: [ service",old_obs//nb_clouds,"cloud",old_obs%nb_clouds,"]\naction :", action,"\nnew obs: [ service",obs//nb_clouds,"cloud",obs%nb_clouds,"]\nreward :", reward ,"\terminated :", terminated )
    print("_____________________________________________________")
print("acc_rew:",acc_rew)

print("Selected services : ", env.composition.print_composition())